In [1]:
#!pip install faiss-cpu
#!pip install sentence-transformers

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -- ------------------------------------- 1.3/18.9 MB 9.5 MB/s eta 0:00:02
   ------- -------------------------------- 3.7/18.9 MB 9.5 MB/s eta 0:00:02
   ------------- -------------------------- 6.6/18.9 MB 11.2 MB/s eta 0:00:02
   ------------------- -------------------- 9.4/18.9 MB 11.7 MB/s eta 0:00:01
   -------------------------- ------------- 12.6/18.9 MB 12.5 MB/s eta 0:00:01
   --------------------------------- ------ 15.7/18.9 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 13.5 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 13.1 MB/s  0:00:01


In [22]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)
from sentence_transformers import SentenceTransformer
import faiss

In [8]:
df = pd.read_csv("sample_text.csv")
print(df.shape)
print(df.text)
df

(8, 2)
0                                          Meditation and yoga can improve mental health
1                       Fruits, whole grains and vegetables helps control blood pressure
2                                      These are the latest fashion trends for this week
3                                      Vibrant color jeans for male are becoming a trend
4                                                     The concert starts at 7 PM tonight
5                        Navaratri dandiya program at Expo center in Mumbai this october
6                                      Exciting vacation destinations for your next trip
7    Maldives and Srilanka are gaining popularity in terms of low budget vacation places
Name: text, dtype: object


,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event
5,Navaratri dandiya program at Expo center in Mumbai this october,Event
6,Exciting vacation destinations for your next trip,Travel
7,Maldives and Srilanka are gaining popularity in terms of low budget vacation places,Travel


### Create source embeddings for the text column

In [71]:
#encoder = SentenceTransformer("all-mpnet-base-v2") #->Baseline reference 
encoder = SentenceTransformer("intfloat/e5-base-v2") #->Balanced performance
#encoder = SentenceTransformer("LaBSE") #->Multilingual tasks
vectors = encoder.encode(df.text)

In [72]:
print(vectors.shape)
vectors

(8, 768)


array([[ 9.4977114e-03, -1.7513877e-02, -3.5399146e-02, ...,
        -1.7197413e-02,  2.4111630e-02, -1.1338668e-02],
       [-4.2614728e-02, -2.2998353e-02, -2.5824152e-02, ...,
        -1.2686604e-02,  7.1906280e-03, -1.4153968e-02],
       [-3.0610496e-03, -6.2897779e-02, -4.3965012e-02, ...,
         1.6430689e-03,  1.8436858e-02,  2.2844408e-02],
       ...,
       [-1.9174362e-02, -3.2043166e-02,  1.7762861e-04, ...,
        -2.2575036e-02,  2.9407699e-02,  2.4151902e-03],
       [-2.0260718e-02, -5.8120616e-02, -2.2026623e-02, ...,
        -1.0610397e-03,  3.7387758e-02, -1.6734654e-02],
       [-2.5707457e-05, -6.1082710e-02, -1.6585749e-02, ...,
        -4.1112229e-02,  6.0512614e-02,  3.3945199e-03]], dtype=float32)

### Build a FAISS Index for vectors

In [73]:
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000020B65EF5BF0> >

### Searching similar vectors

In [75]:
#search_query = "An apple a day keeps the doctor away"
search_query = "I want to buy a car"
vec = encoder.encode(search_query)
vec.shape

(768,)

In [76]:
# Converting to 2-D array as index.search expects a 2-D array

svec = np.array(vec).reshape(1,-1)
svec.shape

(1, 768)

In [77]:
distances, array_position = index.search(svec, k=2) #k=2 means the number of required vectors
print(distances)
print(array_position)

[[0.5420191 0.5480564]]
[[3 6]]


In [78]:
df.loc[array_position[0]]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
6,Exciting vacation destinations for your next trip,Travel
